In [1]:
import pandas as pd
import xml.etree.ElementTree as ET


In [3]:
# Parse XML file
tree = ET.parse("Slack/pythondev_2017.xml")
root = tree.getroot()


In [4]:
messages = []

for msg in root.findall("message"):
    conv_id = msg.attrib.get("conversation_id")
    ts = msg.find("ts").text if msg.find("ts") is not None else None
    user = msg.find("user").text if msg.find("user") is not None else None
    text = msg.find("text").text if msg.find("text") is not None else None

    messages.append({
        "conversation_id": conv_id,
        "timestamp": ts,
        "user": user,
        "text": text
    })

df_2017 = pd.DataFrame(messages)


In [5]:
tree = ET.parse("Slack/pythondev_2018.xml")
root = tree.getroot()
messages = []

for msg in root.findall("message"):
    conv_id = msg.attrib.get("conversation_id")
    ts = msg.find("ts").text if msg.find("ts") is not None else None
    user = msg.find("user").text if msg.find("user") is not None else None
    text = msg.find("text").text if msg.find("text") is not None else None

    messages.append({
        "conversation_id": conv_id,
        "timestamp": ts,
        "user": user,
        "text": text
    })

df_2018 = pd.DataFrame(messages)

In [7]:
tree = ET.parse("Slack/pythondev_2019.xml")
root = tree.getroot()
messages = []

for msg in root.findall("message"):
    conv_id = msg.attrib.get("conversation_id")
    ts = msg.find("ts").text if msg.find("ts") is not None else None
    user = msg.find("user").text if msg.find("user") is not None else None
    text = msg.find("text").text if msg.find("text") is not None else None

    messages.append({
        "conversation_id": conv_id,
        "timestamp": ts,
        "user": user,
        "text": text
    })

df_2019 = pd.DataFrame(messages)

In [8]:
# df = pd.concat(df_2017,df_2018)
df = pd.concat([df_2017, df_2018, df_2019])

In [ ]:
df.count()

conversation_id    106262
timestamp          106262
user               106262
text               106262
dtype: int64

In [9]:
df.to_csv("slack_messages.csv", index=False)


In [4]:
#Find 1000 nearest neighbours of this message from slack_messages.csv "@Alex, can you fix this production issue by next week?" Use BERT or word embeddings
df = pd.read_csv("slack_messages.csv")


NameError: name 'pd' is not defined

In [6]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

In [8]:
df = pd.read_csv("slack_messages.csv")

In [1]:
!pip install transformers torch

In [2]:
from transformers import BertModel, BertTokenizer
import torch

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
df=pd.read_csv("slack_messages.csv")

In [15]:
df.head()

,conversation_id,timestamp,user,text
0,1,2017-06-16T10:51:34.290598,Melvin,Is it possible to switch between conda and vir...
1,1,2017-06-16T10:55:34.384740,Beula,"What is it that ""requires virtualenv"", most th..."
2,1,2017-06-16T11:05:27.626658,Melvin,<@Beula> zappa …. at least if you don’t want t...
3,1,2017-06-16T11:12:26.790435,Johana,i don’t see why you couldn’t use a virtualenv ...
4,1,2017-06-16T11:12:47.798620,Johana,what have you tried so far?


In [17]:
!pip install sentence_transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 488.0/488.0 kB 15.4 MB/s eta 0:00:00


In [18]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import re

# Load your Slack CSV data
df = pd.read_csv('slack_messages.csv')

# Display basic info
print(f"Total messages: {len(df)}")
print(f"Columns: {df.columns.tolist()}\n")

# Clean and prepare text data
def clean_text(text):
    """Clean message text"""
    if pd.isna(text):
        return ""
    # Remove excessive whitespace
    text = ' '.join(text.split())
    return text

df['cleaned_text'] = df['text'].apply(clean_text)

# Remove empty messages
df = df[df['cleaned_text'].str.len() > 0].reset_index(drop=True)
print(f"Messages after removing empty: {len(df)}\n")

# Load pre-trained sentence transformer model
# Using a model optimized for semantic similarity
print("Loading sentence transformer model...")
model = SentenceTransformer('all-MiniLM-L6-v2')  # Fast and efficient
# Alternative: 'all-mpnet-base-v2' (more accurate but slower)

# Define reference task messages
# These are examples of messages that clearly contain tasks
reference_tasks = [
    "@Alex can you fix the production issue by tomorrow?",
    "Can someone review this pull request?",
    "Please update the documentation by end of week",
    "Need help debugging this issue",
    "Could you deploy this to staging?",
    "Assign this ticket to the backend team",
    "Can you investigate why the API is slow?",
    "Please create a ticket for this bug",
    "Who can take ownership of this feature?",
    "Need someone to test this before release"
]

print("Encoding reference task messages...")
reference_embeddings = model.encode(reference_tasks, show_progress_bar=True)

# Encode all Slack messages
print("\nEncoding all Slack messages (this may take a few minutes)...")
message_embeddings = model.encode(df['cleaned_text'].tolist(),
                                   show_progress_bar=True,
                                   batch_size=32)

# Calculate similarity scores
print("\nCalculating similarity scores...")
# Compute average similarity to all reference tasks
similarities = cosine_similarity(message_embeddings, reference_embeddings)
df['task_score'] = similarities.mean(axis=1)

# Additional heuristic features to boost task identification
def has_task_indicators(text):
    """Check for common task indicators"""
    text_lower = text.lower()

    # Task verbs and phrases
    task_patterns = [
        r'\b(can you|could you|please|need to|have to|must)\b',
        r'\b(fix|update|create|deploy|review|investigate|debug|test|assign)\b',
        r'\b(by tomorrow|by \w+day|by end of|deadline|asap|urgent)\b',
        r'\b(issue|bug|ticket|task|todo|pr|pull request)\b',
        r'@\w+',  # Mentions often indicate task assignments
        r'\?',    # Questions often contain requests
    ]

    score = 0
    for pattern in task_patterns:
        if re.search(pattern, text_lower):
            score += 1

    return score

df['heuristic_score'] = df['cleaned_text'].apply(has_task_indicators)

# Combine scores (weighted)
df['combined_score'] = (df['task_score'] * 0.7) + (df['heuristic_score'] * 0.05)

# Sort by combined score and get top 1000
df_sorted = df.sort_values('combined_score', ascending=False)
top_1000_tasks = df_sorted.head(1000)

# Save results
print("\nSaving results...")
top_1000_tasks.to_csv('top_1000_task_messages.csv', index=False)

# Display sample results
print("\n" + "="*80)
print("TOP 20 TASK-LIKE MESSAGES:")
print("="*80)
for idx, row in top_1000_tasks.head(20).iterrows():
    print(f"\nScore: {row['combined_score']:.3f} | User: {row['user']}")
    print(f"Message: {row['text'][:150]}...")
    print("-"*80)

# Statistics
print("\n" + "="*80)
print("STATISTICS:")
print("="*80)
print(f"Total messages analyzed: {len(df)}")
print(f"Top 1000 task messages saved to: top_1000_task_messages.csv")
print(f"\nScore distribution in top 1000:")
print(top_1000_tasks['combined_score'].describe())

# Optional: Find messages most similar to your specific example
print("\n" + "="*80)
print("MESSAGES MOST SIMILAR TO YOUR EXAMPLE:")
print("="*80)
specific_query = "@Alex can you fix the production issue by tomorrow?"
query_embedding = model.encode([specific_query])
specific_similarities = cosine_similarity(message_embeddings, query_embedding).flatten()
df['specific_similarity'] = specific_similarities

top_similar = df.nlargest(20, 'specific_similarity')
for idx, row in top_similar.iterrows():
    print(f"\nSimilarity: {row['specific_similarity']:.3f} | User: {row['user']}")
    print(f"Message: {row['text'][:150]}...")
    print("-"*80)

# Save the full dataset with scores for further analysis
df_sorted.to_csv('all_messages_with_scores.csv', index=False)
print("\n✓ All messages with scores saved to: all_messages_with_scores.csv")

Total messages: 106262
Columns: ['conversation_id', 'timestamp', 'user', 'text']

Messages after removing empty: 105237

Loading sentence transformer model...
Encoding reference task messages...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.41it/s]



Encoding all Slack messages (this may take a few minutes)...


Batches: 100%|██████████| 3289/3289 [00:40<00:00, 81.53it/s] 



Calculating similarity scores...

Saving results...

TOP 20 TASK-LIKE MESSAGES:

Score: 0.373 | User: Suellen
Message: <@Carroll> looks ok, you really have to debug the issue itself...
--------------------------------------------------------------------------------

Score: 0.361 | User: Emelda
Message: <@Henrietta> - could you put that code up on github for us to review? I think we would be more helpful if we actually saw the code. And as an encourag...
--------------------------------------------------------------------------------

Score: 0.358 | User: Emelda
Message: <@Coletta> can you provide with the config? It’s fairly hard to tell without seeing the package config or such. I’m sure many here might find it easie...
--------------------------------------------------------------------------------

Score: 0.351 | User: Ashley
Message: <@Karl> your questions is very vague. Can you be more specific with the issue you're having?...
-----------------------------------------------------